In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip uninstall -y numpy pandas pyarrow datasets

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
Found existing installation: pyarrow 19.0.1
Uninstalling pyarrow-19.0.1:
  Successfully uninstalled pyarrow-19.0.1
Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1


In [3]:
!pip install numpy==1.26.4 pandas==2.1.4 pyarrow==14.0.2 datasets==2.17.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 104.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 118.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 55.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully uninstalled fsspec-2025.10.0
  Attempting uni

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np

2025-12-25 09:40:23.550570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766655623.571206     129 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766655623.577520     129 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Load dataset
dataset_fabsa = load_dataset("jordiclive/fabsa")
train_ds = dataset_fabsa["train"]
test_ds = dataset_fabsa["test"]

Generating train split:   0%|          | 0/7930 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1057 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [5]:
# Extract unique Aspect Labels (ignoring sentiment)
all_aspects = set()

# We iterate over the raw list to avoid tensor errors
for label_entry in train_ds['labels']:
    for aspect, sentiment in label_entry:
        all_aspects.add(aspect)

aspect_list = sorted(list(all_aspects))
num_labels = len(aspect_list)
label2id = {l: i for i, l in enumerate(aspect_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Found {num_labels} unique categories: {aspect_list}")

Found 12 unique categories: ['Account management: Account access', 'Company brand: Competitor', 'Company brand: General satisfaction', 'Company brand: Reviews', 'Logistics rides: Speed', 'Online experience: App website', 'Purchase booking experience: Ease of use', 'Staff support: Attitude of staff', 'Staff support: Email', 'Staff support: Phone', 'Value: Discounts promotions', 'Value: Price value for money']


In [6]:
# Data Processing (Multi-Hot Encoding)
def encode_data(example):
    # Create a vector of zeros [0, 0, ... 0]
    vec = [0.0] * num_labels 
    
    # Loop through labels, get aspect, ignore sentiment
    for aspect, sentiment in example["labels"]:
        if aspect in label2id:
            idx = label2id[aspect]
            vec[idx] = 1.0
            
    # Tokenize text
    enc = tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)
    
    # Add labels to the encoding
    enc["labels"] = vec
    return enc

In [7]:
# Initialize Tokenizer
model_name = "albert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

In [8]:
# Apply processing
train_ds = train_ds.map(encode_data, batched=False)
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Apply processing
test_ds = test_ds.map(encode_data, batched=False)
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/7930 [00:00<?, ? examples/s]

Map:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [9]:
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False)

In [10]:
# Load model
bert = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Apply LoRA
lora_cfg = LoraConfig(
    r=16,          # Rank (Paper uses full fine-tuning, but r=16 is good for LoRA)
    lora_alpha=32,
    target_modules=["query", "key", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

In [12]:
model = get_peft_model(bert, lora_cfg)
model.to(device)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): AlbertForSequenceClassification(
      (albert): AlbertModel(
        (embeddings): AlbertEmbeddings(
          (word_embeddings): Embedding(30000, 128, padding_idx=0)
          (position_embeddings): Embedding(512, 128)
          (token_type_embeddings): Embedding(2, 128)
          (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0, inplace=False)
        )
        (encoder): AlbertTransformer(
          (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
          (albert_layer_groups): ModuleList(
            (0): AlbertLayerGroup(
              (albert_layers): ModuleList(
                (0): AlbertLayer(
                  (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
                  (attention): AlbertSdpaAttention(
                    (query): lora.Linear(
                      (base_layer)

In [13]:
# Optimizer
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [14]:
loss_fn = nn.BCEWithLogitsLoss()

In [15]:
epochs = 10 # FABSA paper suggests training until convergence (usually 5-10 epochs)

print("\nStarting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].float().to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")


Starting Training...


Epoch 1: 100%|██████████| 496/496 [01:17<00:00,  6.42it/s]


Epoch 1 Loss: 0.3429


Epoch 2: 100%|██████████| 496/496 [01:16<00:00,  6.48it/s]


Epoch 2 Loss: 0.2810


...

Epoch 8 Loss: 0.1591


Epoch 9: 100%|██████████| 496/496 [01:16<00:00,  6.46it/s]


Epoch 9 Loss: 0.1527


Epoch 10: 100%|██████████| 496/496 [01:16<00:00,  6.47it/s]

Epoch 10 Loss: 0.1456


In [16]:
model.eval()
y_true = []
y_pred = []

print("\nStarting Evaluation...")
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Sigmoid activation -> Probability
        probs = torch.sigmoid(logits)
        
        # Threshold at 0.5 (Standard for multi-label)
        preds = (probs > 0.3).int().cpu().numpy()
        labels = batch["labels"].cpu().numpy()
        
        y_true.extend(labels)
        y_pred.extend(preds)

# Calculate Metrics
weighted_f1 = f1_score(y_true, y_pred, average="weighted")
precision = precision_score(y_true, y_pred, average="micro")
recall = recall_score(y_true, y_pred, average="micro")


y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

print(f"Weighted F1:  {weighted_f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")


Starting Evaluation...


100%|██████████| 100/100 [00:06<00:00, 14.97it/s]

Weighted F1:  0.7650
Precision: 0.7221
Recall:    0.8072


In [17]:
y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

sample_accuracies = []
for t, p in zip(y_true_arr, y_pred_arr):
    correct = (t * p).sum()               # count correctly predicted labels
    total = t.sum()                       # total actual labels for that sample
    if total == 0:                         # if no true labels exist
        sample_accuracies.append(1.0)      
    else:
        sample_accuracies.append(correct / total)

overall_label_accuracy = np.mean(sample_accuracies)
print(f"Label-wise Sample Accuracy: {overall_label_accuracy:.4f}")


Label-wise Sample Accuracy: 0.8307
